In [1]:
import os
import shutil
import random

source = "./animals"

train_dir = "./animals/train"
test_dir = "./animals/test"

# Create train/test folders
for folder in ["Cat", "Dog"]:
    os.makedirs(os.path.join(train_dir, folder), exist_ok=True)
    os.makedirs(os.path.join(test_dir, folder), exist_ok=True)

# Split each class
for class_name in ["cat", "dog"]:

    class_path = os.path.join(source, class_name)

    images = [
        f for f in os.listdir(class_path)
        if f.lower().endswith(
            (".jpg", ".jpeg", ".png", ".bmp", ".webp")
        )
    ]

    random.shuffle(images)

    split = int(0.8 * len(images))

    train_images = images[:split]
    test_images = images[split:]

    for image in train_images:
        shutil.copy(
            os.path.join(class_path, image),
            os.path.join(train_dir, class_name, image)
        )

    for image in test_images:
        shutil.copy(
            os.path.join(class_path, image),
            os.path.join(test_dir, class_name, image)
        )

    print(class_name)
    print("Training:", len(train_images))
    print("Testing :", len(test_images))

cat
Training: 10392
Testing : 2598
dog
Training: 10375
Testing : 2594


In [1]:
import os
import warnings
from PIL import Image

def clean_dataset_strict(root_dir):
    removed = 0
    for subdir, _, files in os.walk(root_dir):
        for file in files:
            path = os.path.join(subdir, file)
            with warnings.catch_warnings():
                warnings.filterwarnings("error")  # turn warnings into exceptions
                try:
                    img = Image.open(path)
                    img.verify()
                except Exception as e:
                    print(f"Removing: {path} -> {e}")
                    os.remove(path)
                    removed += 1
    print(f"Done. Removed {removed} files.")

clean_dataset_strict("./animals/train")
clean_dataset_strict("./animals/test")

Removing: ./animals/train\Dog\9041.jpg -> Truncated File Read
Done. Removed 1 files.
Done. Removed 0 files.


In [2]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())

2.13.0+cu126
False


In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision

In [4]:
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
import torchvision.transforms as transforms

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((128, 128)),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])


trainset = ImageFolder(
    root="./animals/train",
    transform=transform
)

testset = ImageFolder(
    root="./animals/test",
    transform=transform
)

trainloader = DataLoader(
    trainset,
    batch_size=32,
    shuffle=True
)

testloader = DataLoader(
    testset, 
    batch_size = 32, 
    shuffle = False
)

In [5]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        self.global_pool = nn.AdaptiveAvgPool2d(1)  # -> (batch, 32, 1, 1)

        self.fc_layer = nn.Sequential(
            nn.Linear(32, 64),
            nn.ReLU(),
            nn.Linear(64, 2)
        )
    def forward(self, x):
        x = self.conv_layers(x)

        x = self.global_pool(x)


        x = x.view(x.size(0), -1)

        x = self.fc_layer(x)

        return x

In [6]:
model = CNN()

In [7]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

In [8]:
epochs = 30

for epoch in range(epochs):
    epoch_train_loss = 0.0

    for images,label in trainloader:
        optimizer.zero_grad()

        output = model.forward(images)
        loss = criterion(output,label) #loss fnx
        loss.backward() #back propagation
        optimizer.step() #update params
        epoch_train_loss +=loss.item()

    print(f"epoch{epoch+1}/{epochs} ==>  loss = {epoch_train_loss/len(trainloader)}")

epoch1/30 ==>  loss = 0.6616177469124028
epoch2/30 ==>  loss = 0.6370948633661977
epoch3/30 ==>  loss = 0.6158741760401079
epoch4/30 ==>  loss = 0.591078402304355
epoch5/30 ==>  loss = 0.5747695415476222
epoch6/30 ==>  loss = 0.5583740045994888
epoch7/30 ==>  loss = 0.5472629452561155
epoch8/30 ==>  loss = 0.5378254413236806
epoch9/30 ==>  loss = 0.5315024987782961
epoch10/30 ==>  loss = 0.5243953864147634
epoch11/30 ==>  loss = 0.5168708432971695
epoch12/30 ==>  loss = 0.5119772810626912
epoch13/30 ==>  loss = 0.505440507185312
epoch14/30 ==>  loss = 0.5008887175424599
epoch15/30 ==>  loss = 0.4955608424581127
epoch16/30 ==>  loss = 0.49203701274998396
epoch17/30 ==>  loss = 0.4822940621295093
epoch18/30 ==>  loss = 0.48010146976621065
epoch19/30 ==>  loss = 0.4761868691370811
epoch20/30 ==>  loss = 0.4684857051497624
epoch21/30 ==>  loss = 0.4672543391769315
epoch22/30 ==>  loss = 0.4630547701024715
epoch23/30 ==>  loss = 0.4572644491254548
epoch24/30 ==>  loss = 0.44923634702040827


In [9]:
correct_label = 0
total = 0

model.eval()
with torch.no_grad():
    for images,label in testloader:
        outputs = model.forward(images)
        _, predicted = torch.max(outputs, 1)
        correct_label += (predicted == label).sum().item()
        total += label.size(0)

print("Accuracy: ",correct_label/total*100)

Accuracy:  81.203007518797
